In [ ]:
"""
Polymarket WebSocket Order Book Dashboard – Auto-Slug
Erkennt automatisch das aktuelle 5-Minuten-Fenster und
wechselt beim Ablauf selbstständig zum nächsten Markt.

pip install requests websockets ipywidgets
"""

import requests, json, time, threading, asyncio
from datetime import datetime, timezone
import ipywidgets as widgets
from IPython.display import display
import websockets
import json

# ── Einstellungen ──────────────────────────────────────────────────────────────
BASE_SLUG   = "btc-updown-5m"   # Prefix – nur das ändern wenn nötig
WINDOW_SECS = 300               # 5 Minuten
SLUG_OFFSET = 0                 # Falls der Timestamp nicht current, sondern
                                # z.B. current+300 ist → auf 300 setzen
# ──────────────────────────────────────────────────────────────────────────────

GAMMA_API = "https://gamma-api.polymarket.com"
WS_URL    = "wss://ws-subscriptions-clob.polymarket.com/ws/market"

CSS = """
<style>
.pm-kacheln     { display:flex; gap:10px; margin-bottom:10px; flex-wrap:wrap; }
.pm-kachel      { background:#f5f5f5; border-radius:8px; padding:8px 16px; min-width:110px; }
.pm-kachel .lbl { font-size:11px; color:#888; margin-bottom:2px; }
.pm-kachel .val { font-size:20px; font-weight:600; font-family:monospace; }
.pm-bid         { color:#1a7f4b; }
.pm-ask         { color:#c0392b; }
.pm-table {
    border-collapse: collapse;
    width: 100%;
    font-family: monospace;
    font-size: 13px;

    table-layout: fixed;   /* WICHTIG */
}

.pm-table th,
.pm-table td {
    text-align: right;
    padding: 2px 6px;
    overflow: hidden;
    white-space: nowrap;
}
.pm-sep         { color:#ccc; text-align:center !important; }
.pm-bar-wrap    { background:#eee; border-radius:4px; height:8px;
                  width:100px; display:inline-block; vertical-align:middle; }
.pm-bar         { height:8px; border-radius:4px; transition:width 0.3s; }
.pm-status      { font-size:11px; color:#aaa; margin-left:10px; }
.pm-dot         { display:inline-block; width:8px; height:8px; border-radius:50%; margin-right:4px; }
.pm-dot-green   { background:#1a7f4b; }
.pm-dot-red     { background:#c0392b; }
.pm-dot-yellow  { background:#e67e22; }
</style>
"""




# - Bitcoin

BTC_PRICE = {"price": None, "ts": 0}
BTC_SNAP = {"price": None}

def fetch_btc_price():
    """holt aktuellen BTC-USD Spotpreis (Binance)"""
    try:
        r = requests.get("https://api.binance.com/api/v3/ticker/price",
                         params={"symbol": "BTCUSDT"},
                         timeout=5)
        return float(r.json()["price"])
    except:
        return None
    
def btc_price_loop():
    while True:
        p = fetch_btc_price()
        if p:
            BTC_PRICE["price"] = p
            BTC_PRICE["ts"] = time.time()
        time.sleep(5)

threading.Thread(target=btc_price_loop, daemon=True).start()

# ── Slug-Logik ─────────────────────────────────────────────────────────────────

def current_slug():
    """Aktuelles 5-Minuten-Fenster als Slug."""
    ts = (int(time.time()) // WINDOW_SECS) * WINDOW_SECS + SLUG_OFFSET
    return f"{BASE_SLUG}-{ts}", ts

def seconds_until_next_window():
    """Sekunden bis zum nächsten Fenster."""
    now = int(time.time())
    return WINDOW_SECS - (now % WINDOW_SECS)

def find_valid_slug():
    """
    Probiert current window und current+1 window aus –
    manchmal ist der Markt schon für das nächste Fenster aktiv.
    """
    for offset in [0, WINDOW_SECS, -WINDOW_SECS]:
        ts = (int(time.time()) // WINDOW_SECS) * WINDOW_SECS + SLUG_OFFSET + offset
        slug = f"{BASE_SLUG}-{ts}"
        try:
            r = requests.get(f"{GAMMA_API}/events", params={"slug": slug}, timeout=8)
            data = r.json()
            if data and data[0].get("markets"):
                return slug, ts, data[0]
        except Exception:
            pass
    return None, None, None

# ── REST: Markt-Daten laden ────────────────────────────────────────────────────

def get_markets_from_event(event):
    markets = []
    for m in event.get("markets", []):
        ids = json.loads(m.get("clobTokenIds", "[]"))
        q   = m["question"]
        markets.append({
            "question": q,
            "yes_id":   ids[0] if len(ids) > 0 else None,
            "no_id":    ids[1] if len(ids) > 1 else None,
        })
    return markets

# ── HTML-Bausteine ─────────────────────────────────────────────────────────────

def kachel_html(label, value, cls=""):
    return (f'<div class="pm-kachel"><div class="lbl">{label}</div>'
            f'<div class="val {cls}">{value}</div></div>')

def bar_html(pct, color):
    return (f'<div class="pm-bar-wrap"><div class="pm-bar" '
            f'style="width:{pct:.0f}%;background:{color}"></div></div>'
            f'&nbsp;<b>{pct:.1f}%</b>')

def book_html(bids, asks, levels=10):

    rows = ""

    # sortieren
    b_sorted = sorted(
        bids,
        key=lambda x: float(x["price"]),
        reverse=True
    )[:levels]

    a_sorted = sorted(
        asks,
        key=lambda x: float(x["price"])
    )[:levels]

    # max size für Balken-Skalierung
    max_bid_size = max(
        [float(x["size"]) for x in b_sorted],
        default=1
    )

    max_ask_size = max(
        [float(x["size"]) for x in a_sorted],
        default=1
    )

    cum_bid = 0
    cum_ask = 0

    n = max(len(b_sorted), len(a_sorted))

    for i in range(n):

        b = b_sorted[i] if i < len(b_sorted) else None
        a = a_sorted[i] if i < len(a_sorted) else None

        # -------------------
        # BID
        # -------------------

        if b:

            bp = float(b["price"]) * 100
            bs = float(b["size"])

            cum_bid += bs

            bw = (bs / max_bid_size) * 100

            bid_html = (
                f'<td class="pm-bid">{bp:.2f}¢</td>'
                f'<td class="pm-bid">{bs:,.1f}</td>'
                f'<td class="pm-bid">{cum_bid:,.1f}</td>'
                f'<td>'
                f'<div style="background:#1a7f4b;height:8px;'
                f'width:{bw:.0f}%;border-radius:4px;"></div>'
                f'</td>'
            )

        else:
            bid_html = "<td></td><td></td><td></td><td></td>"

        # -------------------
        # ASK
        # -------------------

        if a:

            ap = float(a["price"]) * 100
            ass = float(a["size"])

            cum_ask += ass

            aw = (ass / max_ask_size) * 100

            ask_html = (
                f'<td>'
                f'<div style="background:#c0392b;height:8px;'
                f'width:{aw:.0f}%;border-radius:4px;"></div>'
                f'</td>'
                f'<td class="pm-ask">{ass:,.1f}</td>'
                f'<td class="pm-ask">{cum_ask:,.1f}</td>'
                f'<td class="pm-ask">{ap:.2f}¢</td>'
            )

        else:
            ask_html = "<td></td><td></td><td></td><td></td>"

        rows += (
            f'<tr>'
            f'{bid_html}'
            f'<td class="pm-sep">│</td>'
            f'{ask_html}'
            f'</tr>'
        )

    return (
        f'<table class="pm-table">'
        f'<tr>'
        f'<th>Bid</th>'
        f'<th>Size</th>'
        f'<th>Cum</th>'
        f'<th>Depth</th>'
        f'<th></th>'
        f'<th>Depth</th>'
        f'<th>Size</th>'
        f'<th>Cum</th>'
        f'<th>Ask</th>'
        f'</tr>'
        f'{rows}'
        f'</table>'
    )

def render_market(kachel_w, book_w, bids_dict, asks_dict, last_trade=None):
    bids = list(bids_dict.values())
    asks = list(asks_dict.values())
    b_s  = sorted(bids, key=lambda x: float(x["price"]), reverse=True)
    a_s  = sorted(asks, key=lambda x: float(x["price"]))
    bb   = float(b_s[0]["price"]) * 100 if b_s else None
    ba   = float(a_s[0]["price"]) * 100 if a_s else None
    sp   = round(ba - bb, 2) if bb and ba else None
    mid = round((bb + ba) / 2, 1) if bb and ba else None
    col  = "#1a7f4b" if (mid or 0) >= 70 else "#e67e22" if (mid or 0) >= 40 else "#c0392b"
    tr   = f'{last_trade*100:.2f}¢' if last_trade else "–"

    snap    = BTC_SNAP.get("price")
    btc_now = BTC_PRICE.get("price")
    snap_html = ""
    if snap and btc_now:
        diff  = btc_now - snap
        color = "#1a7f4b" if diff >= 0 else "#c0392b"
        arrow = "▲" if diff >= 0 else "▼"
        snap_html = kachel_html(
            "BTC Start",
            f'${snap:,.0f} <span style="color:{color};font-size:13px">'
            f'{arrow} {abs(diff):,.0f}</span>'
        )

    kachel_w.value = (
        '<div class="pm-kacheln">'
        + snap_html
        + kachel_html("Best Bid",   f"{bb:.2f}¢" if bb else "–", "pm-bid")
        + kachel_html("Best Ask",   f"{ba:.2f}¢" if ba else "–", "pm-ask")
        + kachel_html("Spread",     f"{sp:.2f}¢" if sp else "–")
        + kachel_html("Last Trade", tr)
        + (f'<div class="pm-kachel"><div class="lbl">Implied Prob.</div>'
           f'<div class="val">{bar_html(mid, col)}</div></div>' if mid else '')
        + '</div>'
    )
    book_w.value = book_html(bids, asks)

# ── WebSocket-Loop ─────────────────────────────────────────────────────────────

async def ws_loop(token_map, widget_map, status_w, title_w, countdown_w, stop_event):

    state = {i: {"bids": {}, "asks": {}, "last_trade": None}     
             for i in widget_map}
    
    while not stop_event.is_set():
        status_w.value = ('<span class="pm-status">'
                          '<span class="pm-dot pm-dot-yellow"></span>Verbinde…</span>')
        try:
            async with websockets.connect(WS_URL, ping_interval=20, ping_timeout=10) as ws:
                await ws.send(json.dumps({
                    "assets_ids": list(token_map.keys()),
                    "type": "market"
                }))
                status_w.value = ('<span class="pm-status">'
                                  '<span class="pm-dot pm-dot-green"></span>'
                                  'Verbunden · warte auf Tick…</span>')

                while not stop_event.is_set():
                    # Countdown bis Fenster-Ende
                    secs_left = seconds_until_next_window()
                    mm, ss    = divmod(secs_left, 60)
                    countdown_w.value = (
                        f'<span style="font-size:12px;color:#888">'
                        f'⏱ Fenster endet in <b>{mm:02d}:{ss:02d}</b></span>'
                    )
                    # ── NEU: BTC Spot-Preis ───────────────────────────────────
                    btc_p = BTC_PRICE.get("price")
                    btc_age = time.time() - BTC_PRICE.get("ts", 0)
                    if btc_p and btc_age < 15:
                        btc_w.value = (
                            f'<span style="font-size:13px;font-weight:600;'
                            f'font-family:monospace;color:#f0a500">'
                            f'₿ {btc_p:,.2f} USD</span>'
                        )
                    else:
                        btc_w.value = '<span style="font-size:12px;color:#aaa">₿ –</span>'

                    # Fenster abgelaufen → neu laden
                    if secs_left <= 1:
                        status_w.value = ('<span class="pm-status">'
                                          '<span class="pm-dot pm-dot-yellow"></span>'
                                          'Fenster läuft ab – lade neuen Markt…</span>')
                        await asyncio.sleep(2)
                        return "reload"   # Signal an äußere Schleife

                    try:
                        raw = await asyncio.wait_for(ws.recv(), timeout=1.0)
                    except asyncio.TimeoutError:
                        continue

                    msgs = json.loads(raw)
                    if not isinstance(msgs, list):
                        msgs = [msgs]

                    for msg in msgs:
                        etype    = msg.get("event_type", "")
                        asset_id = msg.get("asset_id", "")
                        idx      = token_map.get(asset_id)
                        if idx is None:
                            continue
                        s = state[idx]

                        if etype == "book":
                            s["bids"] = {b["price"]: b for b in msg.get("bids", [])}
                            s["asks"] = {a["price"]: a for a in msg.get("asks", [])}
                        elif etype == "price_change":
                            for c in msg.get("changes", []):
                                price = c["price"]
                                size  = float(c["size"])
                                book  = s["bids"] if c["side"].upper() == "BUY" else s["asks"]
                                if size == 0:
                                    book.pop(price, None)
                                else:
                                    book[price] = {"price": price, "size": str(size)}
                        elif etype == "last_trade_price":
                            s["last_trade"] = float(msg.get("price", 0))
                        else:
                            continue

                        kw, bw = widget_map[idx]
                        render_market(kw, bw, s["bids"], s["asks"], s["last_trade"]) 
                        ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
                        status_w.value = (
                            f'<span class="pm-status">'
                            f'<span class="pm-dot pm-dot-green"></span>'
                            f'Live · {ts}</span>'
                        )

        except Exception as e:
            status_w.value = (f'<span class="pm-status">'
                              f'<span class="pm-dot pm-dot-red"></span>'
                              f'Verbindungsfehler: {e} – reconnect in 3s…</span>')
            await asyncio.sleep(3)


def master_loop(widget_map_ref, status_w, title_w, countdown_w,
                kachel_row_w, book_row_w, question_row_w, stop_event):
    """
    Äußere Schleife: erkennt Fenster-Wechsel, lädt neuen Slug,
    baut token_map neu und startet WebSocket neu.
    """
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    while not stop_event.is_set():
        # Aktuellen gültigen Slug finden
        slug, ts, event = find_valid_slug()
        if not slug:
            status_w.value = ('<span class="pm-status">'
                              '<span class="pm-dot pm-dot-red"></span>'
                              'Kein aktiver Markt gefunden – retry in 5s…</span>')
            time.sleep(5)
            continue

        BTC_SNAP["price"] = BTC_PRICE.get("price")

        markets = get_markets_from_event(event)
        title_w.value = (
            f'<div style="font-size:15px;font-weight:600;margin-bottom:4px">'
            f'{event.get("title", slug)}</div>'
            f'<div style="font-size:11px;color:#aaa;margin-bottom:8px">'
            f'Slug: {slug}</div>'
        )

        # Token-Map + Widget-Map aufbauen
        token_map  = {}
        widget_map = {}
        for i, m in enumerate(markets):
            if not m["yes_id"]:
                continue
            token_map[m["yes_id"]] = i
            widget_map[i] = (kachel_row_w[i], book_row_w[i])
            question_row_w[i].value = (
                f'<div style="font-weight:600;margin:10px 0 4px;font-size:13px">'
                f'{m["question"]}</div>'
            )
            kachel_row_w[i].value = '<div style="color:#aaa;font-size:12px">Warte auf ersten Tick…</div>'
            book_row_w[i].value   = ""

        widget_map_ref[0] = widget_map

        result = loop.run_until_complete(
            ws_loop(token_map, widget_map, status_w, title_w, countdown_w, stop_event)
        )

        if result != "reload" or stop_event.is_set():
            break

        # Kurz warten bis neues Fenster sicher aktiv ist
        time.sleep(3)

    loop.close()

# ── Dashboard aufbauen ─────────────────────────────────────────────────────────

# Initiale Daten laden
slug0, ts0, event0 = find_valid_slug()
if not event0:
    print("❌ Kein aktiver Markt gefunden. SLUG_OFFSET anpassen?")
else:   
    markets0 = get_markets_from_event(event0)

    N = len(markets0)

    title_w     = widgets.HTML("")
    status_w    = widgets.HTML("")
    countdown_w = widgets.HTML("")
    btc_w       = widgets.HTML("")
    btn_stop    = widgets.Button(description="⏹ Stop", button_style="danger",
                                 layout=widgets.Layout(width="90px"))

    # Feste Widget-Slots (max N Märkte)
    question_row_w = [widgets.HTML("") for _ in range(N)]
    kachel_row_w   = [widgets.HTML("") for _ in range(N)]
    book_row_w     = [widgets.HTML("") for _ in range(N)]
    sep_w          = [widgets.HTML('<hr style="border:none;border-top:1px solid #eee;margin:6px 0">')
                      for _ in range(N)]

    rows = []
    for i in range(N):
        rows += [question_row_w[i], kachel_row_w[i], book_row_w[i], sep_w[i]]

    panel = widgets.VBox([
        widgets.HTML(CSS),
        title_w,
        widgets.HBox([btn_stop, countdown_w, btc_w, status_w]),
        widgets.HTML('<hr style="border:none;border-top:1px solid #ddd;margin:8px 0">'),
        *rows,
    ])
    display(panel)

    widget_map_ref = [{}]
    stop_event     = threading.Event()

    def on_stop(_):
        stop_event.set()
        btn_stop.disabled = True
        status_w.value = ('<span class="pm-status">'
                          '<span class="pm-dot pm-dot-red"></span>Gestoppt.</span>')

    btn_stop.on_click(on_stop)

    thread = threading.Thread(
        target=master_loop,
        args=(widget_map_ref, status_w, title_w, countdown_w,
              kachel_row_w, book_row_w, question_row_w, stop_event),
        daemon=True,
    )
    thread.start()

In [ ]:
from web3 import Web3
import re

slug = current_slug()[0]
print(f"Suche Chainlink-Feed-Daten für Fenster-Slug: {slug}")
window_ts = int(re.search(r"-(\d+)$", slug).group(1))

ABI = [
    {"inputs": [{"name": "_roundId", "type": "uint80"}],
     "name": "getRoundData",
     "outputs": [{"name": "roundId", "type": "uint80"}, {"name": "answer", "type": "int256"},
                 {"name": "startedAt", "type": "uint256"}, {"name": "updatedAt", "type": "uint256"},
                 {"name": "answeredInRound", "type": "uint80"}],
     "type": "function"},
    {"inputs": [], "name": "latestRoundData",
     "outputs": [{"name": "roundId", "type": "uint80"}, {"name": "answer", "type": "int256"},
                 {"name": "startedAt", "type": "uint256"}, {"name": "updatedAt", "type": "uint256"},
                 {"name": "answeredInRound", "type": "uint80"}],
     "type": "function"}
]

for rpc_url in [
    "https://polygon-bor-rpc.publicnode.com",
    "https://polygon-rpc.com",
    "https://rpc.ankr.com/polygon",
]:
    try:
        w3 = Web3(Web3.HTTPProvider(rpc_url, request_kwargs={"timeout": 10}))
        feed = w3.eth.contract(
            address="0xc907E116054Ad103354f2D350FD2514433D57F6f",
            abi=ABI
        )
        latest = feed.functions.latestRoundData().call()
        break
    except Exception:
        continue

phase_id = latest[0] >> 64
aggregator_round = latest[0] & 0xFFFFFFFFFFFFFFFF

# Finde die Runde, die zu window_ts aktiv war:
# updatedAt[i] <= window_ts < updatedAt[i+1]
found_round = None
found_data = None

for i in range(aggregator_round, max(aggregator_round - 500, 0), -1):
    round_id = (phase_id << 64) | i
    try:
        data = feed.functions.getRoundData(round_id).call()
        updated_at = data[3]  # ← updatedAt statt startedAt

        if updated_at <= window_ts:
            # Prüfe ob die NÄCHSTE Runde erst nach window_ts liegt
            next_round_id = (phase_id << 64) | (i + 1)
            try:
                next_data = feed.functions.getRoundData(next_round_id).call()
                next_updated_at = next_data[3]
                if next_updated_at > window_ts:
                    found_round = i
                    found_data = data
                    break
                # Falls nächste Runde auch noch <= window_ts → weiter suchen
            except:
                # Keine nächste Runde → aktuelle ist die letzte
                found_round = i
                found_data = data
                break
    except:
        continue

if found_data:
    strike = found_data[1] / 1e8
    print(f"Strike (Price to Beat): ${strike:,.2f}")
    print(f"updatedAt:              {found_data[3]}  (window_ts: {window_ts})")
    print(f"Differenz:              {window_ts - found_data[3]}s vor Fensterstart")
else:
    print("Keine passende Runde gefunden.")